# Milestone 4: speculative decoding upgrade (prompt-lookup drafter + cost model)
Kaggle: **GPU T4 x2, Internet ON**, Run All. About 15-20 min. Uses the final adapters from HuggingFace, so nothing is retrained.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
TARGET_ADAPTER = 'usmanbari2005/trialmatch-qwen25-1p5b-lora-v2'
DRAFT_ADAPTER = 'usmanbari2005/trialmatch-qwen25-0p5b-draft-lora'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf trial-eligibility-transformer
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd /kaggle/working/trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q tests
!python scripts/prepare_data.py | tail -2

## Speculative decoding: prompt lookup vs base draft vs tuned draft, k = 2/4/6/8 (fp16), with measured cost model

In [ ]:
!python scripts/bench_spec.py --adapter {TARGET_ADAPTER} --drafts lookup=prompt_lookup base=Qwen/Qwen2.5-0.5B-Instruct tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 2 4 6 8 --dtype float16

## fp32 exactness check (lookup and tuned draft, k=4)

In [ ]:
!python scripts/bench_spec.py --adapter {TARGET_ADAPTER} --drafts lookup=prompt_lookup tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 4 --dtype float32 --n-task 10 --n-summary 3

In [ ]:
!python scripts/make_bench_figures.py
import json
d = json.load(open('results/bench/spec_float16.json'))
print('cost model:', json.dumps(d['cost_model'], indent=1))
for name, r in d['drafts'].items():
    print(name, 'c =', r.get('c'))

## Download `m4_results.zip`

In [ ]:
!zip -qr /kaggle/working/m4_results.zip results/bench results/figures
!ls -la /kaggle/working